In [6]:
# Run this in your terminal or a notebook cell:
# !pip install torch python-chess fastapi uvicorn pydantic

import torch
import torch.nn as nn
import torch.optim as optim
import chess
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

In [7]:
class LightweightChessNet(nn.Module):
    def __init__(self):
        super(LightweightChessNet, self).__init__()
        # A small convolution block to process the 8x8 board
        self.conv = nn.Sequential(
            nn.Conv2d(14, 64, kernel_size=3, padding=1), # 14 channels for pieces/rules
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        # Policy Head: Predicts the best move
        self.policy_head = nn.Sequential(
            nn.Conv2d(64, 2, kernel_size=1),
            nn.BatchNorm2d(2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(2 * 8 * 8, 4096), # 4096 possible moves max
            nn.LogSoftmax(dim=1)
        )
        
        # Value Head: Predicts the game outcome (-1 to 1)
        self.value_head = nn.Sequential(
            nn.Conv2d(64, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.conv(x)
        policy = self.policy_head(x)
        value = self.value_head(x)
        return policy, value

# Initialize the model
model = LightweightChessNet()

In [8]:
def train_model(model, epochs=5):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # For demonstration, we create dummy data representing a board state, 
    # a target policy, and a target game outcome.
    # In a real scenario, this data comes from the MCTS self-play logs.
    dummy_board_state = torch.randn(16, 14, 8, 8) 
    target_policy = torch.randn(16, 4096)
    target_value = torch.randn(16, 1)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # Forward pass
        policy_pred, value_pred = model(dummy_board_state)
        
        # Calculate loss (Mean Squared Error for Value, Cross Entropy for Policy)
        value_loss = nn.MSELoss()(value_pred, target_value)
        policy_loss = -torch.sum(target_policy * policy_pred) / 16
        loss = value_loss + policy_loss
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

train_model(model)

# Save the lightweight model weights for serving
torch.save(model.state_dict(), "lightweight_chess_model.pth")

Epoch 1/5 | Loss: -72.1069
Epoch 2/5 | Loss: -154.4853
Epoch 3/5 | Loss: -235.7532
Epoch 4/5 | Loss: -315.9965
Epoch 5/5 | Loss: -395.8843


In [12]:
def board_to_tensor(board):
    # Simplified feature extraction: convert pieces to a 14x8x8 tensor
    # 12 channels for pieces (6 white, 6 black), 2 for castling/turn rights
    tensor = torch.zeros(1, 14, 8, 8)
    # (Implementation of mapping standard pieces to 8x8 grids goes here)
    return tensor

def get_best_move(board, model):
    model.eval()
    board_tensor = board_to_tensor(board)
    
    with torch.no_grad():
        policy, value = model(board_tensor)
    
    # Filter for legal moves only and pick the one with the highest policy score
    legal_moves = list(board.legal_moves)
    best_move = None
    best_score = -float('inf')
    
    for move in legal_moves:
        # In a real implementation, you map the move to its specific index in the 4096 vector
        move_index = hash(move.uci()) % 4096 
        score = policy[0][move_index].item()
        if score > best_score:
            best_score = score
            best_move = move
            
    return best_move

# Test the inference
test_board = chess.Board()
print("Best move predicted:", get_best_move(test_board, model))

Best move predicted: a2a4


In [10]:
app = FastAPI(title="Lightweight Chess Engine API")

# Load model on startup
model = LightweightChessNet()
model.load_state_dict(torch.load("lightweight_chess_model.pth"))
model.eval()

# Pydantic model for API payload validation
class FENRequest(BaseModel):
    fen: str

@app.post("/predict_move")
def predict_move(request: FENRequest):
    try:
        # Load the board from the FEN string
        board = chess.Board(request.fen)
        
        # Get prediction
        best_move = get_best_move(board, model)
        
        return {
            "fen": request.fen,
            "best_move": best_move.uci() if best_move else "None",
            "status": "success"
        }
    except Exception as e:
        return {"error": str(e), "status": "failed"}

# To run the server, use this command in your terminal:
# uvicorn filename:app --host 0.0.0.0 --port 8000